In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import re
import os
import pandas as pd

In [ ]:
def read_markdown_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def clean_markdown_headers(text):
    # Xóa các dấu ** bao quanh text nằm ngay sau các dấu #
    return re.sub(r'(#+)\s*\*\*(.*?)\*\*', r'\1 \2', text)

In [ ]:
# Cấu hình chunking cho từng loại dữ liệu theo yêu cầu
configs = {
    "Guidelines.md": {
        "document_type": "guideline",
        "headers_to_split_on": [
            ("#", "Heading_I"),
            ("##", "Heading_II"),
            ("###", "Heading_III")
        ]
    },
    "Monographs.md": {
        "document_type": "monograph",
        "headers_to_split_on": [
            ("#", "Heading_I"),
            ("##", "Heading_II")
        ]
    },
    "Appendices_2.md": {
        "document_type": "appendix",
        "headers_to_split_on": [
            ("#", "Heading_I"),
            ("##", "Heading_II")
        ]
    },
    "Appendices_3.md": {
        "document_type": "appendix",
        "headers_to_split_on": [
            ("#", "Heading_I"),
            ("##", "Heading_II"),
            ("###", "Heading_III")
        ]
    }
}

In [ ]:
MAX_TOKENS = 8191 - 50

# Splitter 2: Cắt các đoạn quá dài theo đúng số lượng TOKEN thay vì ký tự
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="text-embedding-3-large",
    chunk_size=MAX_TOKENS,      # Giới hạn TOKEN
    chunk_overlap=0,            # Chồng lấp 0 TOKEN
)

In [ ]:
def process_file_to_chunks(file_path, config):
    raw_text = read_markdown_file(file_path)
    cleaned_text = clean_markdown_headers(raw_text)
    
    # Bước 1: Cắt theo Header và gắn Metadata tương ứng với cấu hình file
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=config["headers_to_split_on"],
        strip_headers=True
    )
    md_header_splits = markdown_splitter.split_text(cleaned_text)
    
    # Bước 2: Cắt nhỏ tiếp các đoạn quá dài (Metadata vẫn được bảo toàn)
    final_splits = text_splitter.split_documents(md_header_splits)
    
    # Bước 3: Ánh xạ thành các trường thống nhất (Unified Keys)
    data_to_save = []
    for chunk in final_splits:
        row = {
            "document_type": config["document_type"],
            "level_1": chunk.metadata.get("Heading_I", None),
            "level_2": chunk.metadata.get("Heading_II", None),
            "level_3": chunk.metadata.get("Heading_III", None),
            "content": chunk.page_content
        }
        data_to_save.append(row)
        
    return data_to_save

In [ ]:
input_dir = r"D:\PharmaRAG-VN\Data\Clean"
output_dir = r"D:\PharmaRAG-VN\Data\Clean"

all_files_chunks = []

for filename, config in configs.items():
    file_path = os.path.join(input_dir, filename)
    if os.path.exists(file_path):
        print(f"--- Đang xử lý {filename} ---")
        chunks = process_file_to_chunks(file_path, config)
        
        # Tạo DataFrame
        df = pd.DataFrame(chunks)
        
        # Đảm bảo thứ tự cột thống nhất
        df = df[["document_type", "level_1", "level_2", "level_3", "content"]]
        
        # Lưu CSV cho từng file riêng biệt
        output_filename = filename.replace(".md", "_chunk.csv")
        output_path = os.path.join(output_dir, output_filename)
        df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"Đã lưu thành công {len(chunks)} chunks ra file: {output_path}\n")
        
        # Gộp tất cả chunks lại để xem tổng quan nếu cần
        all_files_chunks.extend(chunks)
    else:
        print(f"Không tìm thấy file: {file_path}\n")

print(f"Hoàn thành! Tổng cộng có {len(all_files_chunks)} chunks từ tất cả các file.")